# Cold-Chain Shipment Delay Prediction

**AI Services MS2 | Supplementary Practice A | 60 Marks | Exactly 20 Evaluation Checks**

**Candidate notebook — complete the functions marked TODO.**

**Business task.** A cold-chain logistics company transports temperature-sensitive parcels between regions. Dispatch records include temperature, estimated transit duration, and a shipment note written by an operations agent. The distance and parcel weight exist inside the note rather than in ready-to-use numeric columns.

**Objective.** Build a reusable binary-classification notebook. Clean mixed-format dates and inconsistent category labels, extract two decimal measurements from shipment notes with simple regular expressions, engineer safe numerical features, and compare Logistic Regression with Random Forest. Use a probability threshold of 0.40 to label risky shipments.

## 1. Imports, parameters and feature groups

In [ ]:
import re
import joblib
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

TARGET = "late_delivery"

RAW_NUMERIC = ["transit_hours", "temperature_c"]

NUMERIC = ["transit_hours", "temperature_c", "distance_km", "weight_kg",
           "month_number", "departure_hour", "distance_per_hour"]

CATEGORICAL = ["region", "vehicle_type", "route_zone"]

FEATURES = NUMERIC + CATEGORICAL

MONTH_MAP = {"jan": 1, "feb": 2, "mar": 3, "apr": 4, "may": 5,
             "jun": 6, "jul": 7, "aug": 8, "sep": 9, "oct": 10, "nov": 11, "dec": 12}

THRESHOLD = 0.40

## Implementation contract — independently testable functions

**1. `extract_decimal(text: object, field: str) -> float`**  
**Returns:** Return a float from a labeled note field (distance or weight), or np.nan if no match exists.  
• Use re.search with a case-insensitive, simple decimal-number capture; find values after a field label followed by a colon, allowing optional spaces.  
• Examples: distance: 82.5 -> 82.5; WEIGHT : 7.20 -> 7.2; missing field -> NaN. No elaborate grammar or unit parser is required.

In [ ]:
def extract_decimal(text: object, field: str) -> float:
    """Return float parsed from simple 'distance: 12.5' or 'weight: 3.2' text; otherwise NaN."""
    match = re.search(rf"{field}\s*:\s*(\d+(?:\.\d+)?)", str(text), flags=re.I)
    return float(match.group(1)) if match else np.nan
    raise NotImplementedError('Complete extract_decimal')

**2. `load_data(filename: str) -> pd.DataFrame`**  
**Returns:** Return one cleaned DataFrame preserving shipment_id, optional late_delivery, engineered predictors and surviving input order.  
• Read the requested CSV, keep the first row per shipment_id, parse dispatch_ts with errors="coerce", and remove invalid timestamp rows.  
• Coerce transit_hours and temperature_c to numeric; turn positive/negative infinity into missing values. Trim and lowercase three category columns and fill blank/missing with unknown.  
• Apply r"^([+-]?\d+)\.0+$" -> r"\1" to numeric-looking categories such as 2.0. Extract distance_km and weight_kg from shipment_note.  
• Map jan..dec to month_number, extract departure_hour, and calculate distance_per_hour only for positive transit_hours.

In [ ]:
def load_data(filename: str) -> pd.DataFrame:
    """Clean a shipment CSV and preserve valid input order, IDs and optional target."""

    df = pd.read_csv(filename,dtype={'shipment_id':'string'})
    df = df.drop_duplicates(subset=['shipment_id'],keep='first')
    df['dispatch_ts'] = pd.to_datetime(df['dispatch_ts'],format='mixed',errors='coerce',dayfirst=True)
    df = df.dropna(subset=['dispatch_ts'])

    for col in RAW_NUMERIC:
        df[col] = pd.to_numeric(df[col],errors='coerce').replace([np.inf,-np.inf],np.nan)

    for col in CATEGORICAL:
        df[col] = df[col].astype('string').str.strip().str.lower().replace(r'^([+-]?\d+)\.0+$',r'\1',regex=True).replace('',pd.NA).fillna('unknown')

    df["distance_km"] = df["shipment_note"].apply(lambda x: extract_decimal(x, "distance"))
    df["weight_kg"] = df["shipment_note"].apply(lambda x: extract_decimal(x, "weight"))

    df['month_number'] = df['month_text'].astype('string').str.strip().str.lower().map(MONTH_MAP)

    df['departure_hour'] = df['dispatch_ts'].dt.hour

    df["distance_per_hour"] = (df["distance_km"] / df["transit_hours"].where(df["transit_hours"] > 0)).replace([np.inf, -np.inf], np.nan)
    
    return df
    raise NotImplementedError('Complete load_data')

**3. `build_pipeline(algorithm: BaseEstimator) -> Pipeline`**  
**Returns:** Return an unfitted Scikit-learn Pipeline combining preprocessing with the estimator supplied.  
• Create a numeric branch with SimpleImputer(strategy="median") followed by StandardScaler for seven numeric predictors.  
• Create a categorical branch with SimpleImputer(strategy="most_frequent") followed by OneHotEncoder(handle_unknown="ignore") for three categorical predictors.  
• Combine both branches with ColumnTransformer, and append the given estimator as a model step.

In [ ]:
def build_pipeline(algorithm: BaseEstimator) -> Pipeline:
    """Build unfitted mixed numeric/categorical preprocessing followed by the supplied model."""

    numeric_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median")),
                                 ("scaler", StandardScaler())])
    categorical_pipeline = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                                     ("encoder", OneHotEncoder(handle_unknown="ignore"))])
    preprocessor = ColumnTransformer([
        ("numeric", numeric_pipeline, NUMERIC),
        ("category", categorical_pipeline, CATEGORICAL),
    ])
    
    return Pipeline([("preprocessor", preprocessor), ("model", algorithm)])
    raise NotImplementedError('Complete build_pipeline')

**4. `prepare_datasets(train_file: str, test_file: str) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]`**  
**Returns:** Return X_train, X_test, y_train and y_test in that exact order.  
• Call the same loader on both files. Normalize accepted target values yes/no and 1/0 to integer 1/0, removing missing or unsupported labels.  
• Select exactly the ten predictor features. Preserve feature/target alignment and do not fit on held-back records.

In [ ]:
def prepare_datasets(train_file: str, test_file: str) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    """Return X_train, X_test, y_train, y_test with 0/1 targets."""

    train_df, test_df = load_data(train_file), load_data(test_file)
    
    for df in (train_df, test_df):
        df[TARGET] = df[TARGET].astype("string").str.strip().str.lower().map({"yes": 1, "no": 0})
        df.dropna(subset=[TARGET], inplace=True)

    X_train = train_df[FEATURES]
    X_test = test_df[FEATURES]

    y_train = train_df[TARGET].astype(int)
    y_test = test_df[TARGET].astype(int)
    
    return X_train,X_test,y_train,y_test
    
    raise NotImplementedError('Complete prepare_datasets')

**5. `train_models(X_train: pd.DataFrame, y_train: pd.Series) -> dict[str, Pipeline]`**  
**Returns:** Return two separately fitted pipelines keyed logistic_regression and random_forest.  
• Fit LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42).  
• Fit RandomForestClassifier(n_estimators=80, max_depth=9, min_samples_leaf=2, class_weight="balanced", random_state=42, n_jobs=-1).  
• Each algorithm gets its own complete preprocessing pipeline.

In [ ]:
def train_models(X_train: pd.DataFrame, y_train: pd.Series) -> dict[str, Pipeline]:
    """Train separately fitted Logistic Regression and Random Forest pipelines."""

    models = {
        "logistic_regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
        "random_forest": RandomForestClassifier(n_estimators=80, max_depth=9,
            min_samples_leaf=2, class_weight="balanced", random_state=42, n_jobs=-1),
    }
    trained_models = {}
    
    for name, algorithm in models.items():
        pipeline = build_pipeline(algorithm)
        pipeline.fit(X_train, y_train)
        trained_models[name] = pipeline
    
    return trained_models

    
    raise NotImplementedError('Complete train_models')

**6. `evaluate_models(trained_models: dict[str, Pipeline], X_test: pd.DataFrame, y_test: pd.Series) -> pd.DataFrame`**  
**Returns:** Return a comparison DataFrame containing model_name, accuracy, precision, recall, f1.  
• Get the predicted probability for positive class 1 and convert to classes using probability >= 0.40.  
• Compute accuracy, precision, recall, positive-class F1 (zero_division=0). Rank by f1 descending, recall descending, model_name ascending; reset index. Never refit.

In [ ]:
def evaluate_models(trained_models: dict[str, Pipeline], X_test: pd.DataFrame, y_test: pd.Series) -> pd.DataFrame:
    """Return accuracy, precision, recall and F1; rank F1 desc, recall desc, name asc."""

    results = []

    for name,pipeline in trained_models.items():
        postive_idx = list(pipeline.classes_).index(1)
        positive_prob = pipeline.predict_proba(X_test)[:,postive_idx]
        predictions = (positive_prob>=THRESHOLD).astype(int)

        results.append({
            'model_name' : name,
            'accuracy' : accuracy_score(y_test,predictions),
            "precision": precision_score(y_test, predictions, zero_division=0),
            "recall": recall_score(y_test, predictions, zero_division=0),
            "f1": f1_score(y_test, predictions, zero_division=0)
        })
    
    comparison = pd.DataFrame(results)
    return comparison.sort_values(["f1", "recall", "model_name"],ascending=[False, False, True]).reset_index(drop=True)
    
    raise NotImplementedError('Complete evaluate_models')

**7. `select_best_model(comparison: pd.DataFrame, trained_models: dict[str, Pipeline]) -> tuple[str, Pipeline]`**  
**Returns:** Return (best_name, best_model) from the already-ranked comparison.  
• Use the first row of the sorted comparison and return that name with the original fitted pipeline from trained_models.

In [ ]:
def select_best_model(comparison: pd.DataFrame, trained_models: dict[str, Pipeline]) -> tuple[str, Pipeline]:
    """Return name and fitted pipeline in the first ranked comparison row."""

    best_name = str(comparison.iloc[0]["model_name"])
    best_model = trained_models[best_name]
    return best_name, best_model

    raise NotImplementedError('Complete select_best_model')

**8. `predict_delay(model: Pipeline, df: pd.DataFrame) -> pd.DataFrame`**  
**Returns:** Return shipment_id, predicted_late, late_probability in that exact order.  
• Receive cleaned queued records. Use positive-class probability, threshold at 0.40, and round probability to four decimal places.  
• Preserve shipment order. Do not retrain or include raw note/ID fields among model predictors.

In [ ]:
def predict_delay(model: Pipeline, df: pd.DataFrame) -> pd.DataFrame:
    """Return shipment_id, predicted_late, late_probability in cleaned-record order."""
    positive_idx = list(model.classes_).index(1)
    positive_prob = model.predict_proba(df[FEATURES])[:,positive_idx]
    predictions = (positive_prob>=THRESHOLD).astype(int)
    df = pd.DataFrame({
        'shipment_id' : df['shipment_id'].to_numpy(),
        'predicted_late' : predictions,
        'probabilty_late' : np.round(positive_prob,4)
    })
    raise NotImplementedError('Complete predict_delay')

## Training and saved-model prediction

The training and prediction workflows are intentionally separate. Training fits and persists the chosen estimator with its preprocessing. Prediction must load the saved model and process new rows without calling `fit()`.

**`train() -> tuple[str, pd.DataFrame]`** — Return selected model name and its comparison report. Call the preparation, training, evaluation and selection functions; write best_model.pkl and model_comparison.csv.

In [ ]:
def train() -> tuple[str, pd.DataFrame]:
    """Train, evaluate and save best_model.pkl and model_comparison.csv."""
    raise NotImplementedError('Complete train')

**`predict() -> pd.DataFrame`** — Return the new shipment predictions DataFrame. Load shipment_predict.csv through the common loader and reload best_model.pkl. Save shipment_predictions.csv without fitting.

**Required output files:** `best_model.pkl`, `model_comparison.csv`, `shipment_predictions.csv`

In [ ]:
def predict() -> pd.DataFrame:
    """Reload saved pipeline and write shipment_predictions.csv without fitting."""
    raise NotImplementedError('Complete predict')

## 4. Run the workflow

Execute the **training** call first, then the **prediction** call. You can also run these in separate notebook sessions after a model has been saved.

In [ ]:
# Uncomment to train after completing the functions:
# best_name, comparison = train()
# display(comparison)

In [ ]:
# Uncomment to predict after training:
# result = predict()
# display(result.head())